In [1]:
from model_wrapper import *
from sklearn.externals.array_api_extra.testing import override

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 298.13 MB


In [2]:
import tensorflow as tf
import numpy as np
import logging
from tensorflow.keras.preprocessing.image import ImageDataGenerator


tf.get_logger().setLevel('ERROR')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
model_index = 0

train_datagen_augmented = ImageDataGenerator(rotation_range=0.2,
                                             shear_range=0.2,
                                             zoom_range=0.2,
                                             width_shift_range=0.2,
                                             height_shift_range=0.2,
                                             horizontal_flip=True)
def load_data_wrapper(model, augment):
    def load_data(folder_path, label):
        def process_instance(path_tensor):
            actual_path = path_tensor.numpy().decode('utf-8')
            x = get_training_instance(actual_path)
            x = np.transpose(x, (1, 2, 0))
            if augment:
                x = train_datagen_augmented.random_transform(x)
            return x.astype(np.float32)

        image_data = tf.py_function(
            func=process_instance,
            inp=[folder_path],
            Tout=tf.float32
        )

        image_data.set_shape([IMG_HEIGHT, IMG_WIDTH, MIN_IMG_COUNT])

        return image_data, tf.expand_dims(label[model.target_index], axis=-1)
    return load_data


I0000 00:00:1790777314.672343   27497 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:


class CNNModel(MultiLabelModel):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, no_valid=False):
        def create_model():
            model = tf.keras.models.Sequential([
                tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, MIN_IMG_COUNT)),
                tf.keras.layers.Conv2D(128, 10, activation=tf.keras.activations.swish),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.Conv2D(64, 5, padding="same", activation=tf.keras.activations.swish),
                tf.keras.layers.MaxPool2D(pool_size=5),
                tf.keras.layers.Conv2D(32, 3, padding="same", activation=tf.keras.activations.swish),
                tf.keras.layers.Conv2D(32, 3, padding="same", activation=tf.keras.activations.swish),
                tf.keras.layers.MaxPool2D(pool_size=3),
                tf.keras.layers.Conv2D(16, 2, padding="same", activation=tf.keras.activations.swish),
                tf.keras.layers.GlobalAvgPool2D(),
                tf.keras.layers.Dense(3, activation=tf.keras.activations.swish),
                tf.keras.layers.Dense(1, activation="sigmoid")
            ])

            model.compile(
                loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25),
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
                metrics=[tf.keras.metrics.AUC(name='auc')]
            )
            return model
        self.models = [create_model() for _ in range(len(target_columns))]

        self.history = [None for _ in range(len(target_columns))]
        self.target_index = model_index
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False, no_valid=no_valid)

    @override
    def batch_fit(self, training_folders: pd.Series, y_train: np.ndarray, validation_folders: pd.Series, y_validation: np.ndarray):
        train_dataset = tf.data.Dataset.from_tensor_slices((training_folders, y_train))
        train_dataset = train_dataset.map(load_data_wrapper(self, True), num_parallel_calls=tf.data.AUTOTUNE)
        train_dataset = train_dataset.batch(BATCH_SIZE)
        train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

        val_dataset = None
        if validation_folders is not None:
            val_dataset = tf.data.Dataset.from_tensor_slices((validation_folders, y_validation))
            val_dataset = val_dataset.map(load_data_wrapper(self, False), num_parallel_calls=tf.data.AUTOTUNE)
            val_dataset = val_dataset.batch(BATCH_SIZE)
            val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

        for _ in range(len(target_columns)):
            self.history[self.target_index] = (self.models[self.target_index].fit(
                train_dataset,
                epochs=10,
                validation_data=val_dataset,
                verbose=2
            ))

            self.target_index += 1
    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.transpose(x, (0, 2, 3, 1))
        pred_list = [tf.squeeze(m.predict(x, verbose=0)) for m in self.models]
        return np.column_stack(pred_list)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.transpose(x, (1, 2, 0))
        x = np.expand_dims(x, axis=0)
        pred_ = [np.squeeze(m.predict(x, verbose=0)) for m in self.models]
        return np.array(pred_)

In [ ]:
ensemble: List[CNNModel] = []

for p in planes:
    for i in range(2):
        if p == "Axial" and i == 0:
            continue
        ensemble.append(CNNModel(p, i, i))
ensemble.append(CNNModel("Axial"))

I0000 00:00:1790777327.491343   27497 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9709 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Training Size: (36,)
	Validation Size: (20,)
Epoch 1/20


/home/zero/venvs/ds-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1790777349.537095   27593 service.cc:153] XLA service 0x74b97c0316e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790777349.537140   27593 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1790777349.615656   27593 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790777349.986103   27593 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1790777350.012760   27593 dot_merger.cc:481] Merging Dots

3/3 - 208s - 69s/step - auc: 0.3734 - loss: 0.1875 - val_auc: 0.5000 - val_loss: 0.1696
Epoch 2/20
3/3 - 16s - 5s/step - auc: 0.5688 - loss: 0.1724 - val_auc: 0.6429 - val_loss: 0.1701
Epoch 3/20
3/3 - 17s - 6s/step - auc: 0.5938 - loss: 0.1677 - val_auc: 0.6429 - val_loss: 0.1704
Epoch 4/20
3/3 - 16s - 5s/step - auc: 0.7172 - loss: 0.1615 - val_auc: 0.4762 - val_loss: 0.1699
Epoch 5/20
3/3 - 17s - 6s/step - auc: 0.7469 - loss: 0.1560 - val_auc: 0.5714 - val_loss: 0.1693
Epoch 6/20
3/3 - 15s - 5s/step - auc: 0.7422 - loss: 0.1601 - val_auc: 0.5714 - val_loss: 0.1692
Epoch 7/20
3/3 - 16s - 5s/step - auc: 0.7250 - loss: 0.1598 - val_auc: 0.5357 - val_loss: 0.1693
Epoch 8/20
3/3 - 15s - 5s/step - auc: 0.7719 - loss: 0.1521 - val_auc: 0.5536 - val_loss: 0.1692
Epoch 9/20
3/3 - 16s - 5s/step - auc: 0.7703 - loss: 0.1577 - val_auc: 0.5476 - val_loss: 0.1696
Epoch 10/20
3/3 - 15s - 5s/step - auc: 0.7953 - loss: 0.1507 - val_auc: 0.5476 - val_loss: 0.1696
Epoch 11/20
3/3 - 15s - 5s/step - auc:

I0000 00:00:1790777845.440364   27593 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_10471__.48


...

3/3 - 18s - 6s/step - auc: 0.5120 - loss: 0.1758 - val_auc: 0.5286 - val_loss: 0.1727
Epoch 2/20
3/3 - 14s - 5s/step - auc: 0.6842 - loss: 0.1600 - val_auc: 0.6643 - val_loss: 0.1724
Epoch 3/20
3/3 - 12s - 4s/step - auc: 0.6866 - loss: 0.1554 - val_auc: 0.6643 - val_loss: 0.1723
Epoch 4/20
3/3 - 14s - 5s/step - auc: 0.6842 - loss: 0.1516 - val_auc: 0.6643 - val_loss: 0.1723
Epoch 5/20
3/3 - 12s - 4s/step - auc: 0.6986 - loss: 0.1506 - val_auc: 0.6643 - val_loss: 0.1723
Epoch 6/20
3/3 - 12s - 4s/step - auc: 0.7153 - loss: 0.1503 - val_auc: 0.6643 - val_loss: 0.1722
Epoch 7/20
3/3 - 14s - 5s/step - auc: 0.7536 - loss: 0.1449 - val_auc: 0.6929 - val_loss: 0.1722
Epoch 8/20
3/3 - 12s - 4s/step - auc: 0.7632 - loss: 0.1384 - val_auc: 0.6429 - val_loss: 0.1721
Epoch 9/20
3/3 - 12s - 4s/step - auc: 0.7536 - loss: 0.1472 - val_auc: 0.6429 - val_loss: 0.1722
Epoch 10/20
3/3 - 14s - 5s/step - auc: 0.7823 - loss: 0.1362 - val_auc: 0.6429 - val_loss: 0.1721
Epoch 11/20
3/3 - 12s - 4s/step - auc: 0

I0000 00:00:1790785772.780938   27590 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_212001__.48
I0000 00:00:1790785778.312103   27590 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_212001__.48


3/3 - 18s - 6s/step - auc: 0.5837 - loss: 0.1706 - val_auc: 0.6000 - val_loss: 0.1728
Epoch 2/20
3/3 - 13s - 4s/step - auc: 0.7033 - loss: 0.1599 - val_auc: 0.5929 - val_loss: 0.1723
Epoch 3/20
3/3 - 14s - 5s/step - auc: 0.6603 - loss: 0.1543 - val_auc: 0.5500 - val_loss: 0.1720
Epoch 4/20
3/3 - 12s - 4s/step - auc: 0.6890 - loss: 0.1528 - val_auc: 0.5643 - val_loss: 0.1720
Epoch 5/20
3/3 - 14s - 5s/step - auc: 0.7225 - loss: 0.1486 - val_auc: 0.5643 - val_loss: 0.1718
Epoch 6/20
3/3 - 12s - 4s/step - auc: 0.7273 - loss: 0.1462 - val_auc: 0.5000 - val_loss: 0.1718
Epoch 7/20
3/3 - 12s - 4s/step - auc: 0.6818 - loss: 0.1523 - val_auc: 0.6286 - val_loss: 0.1718
Epoch 8/20
3/3 - 15s - 5s/step - auc: 0.7033 - loss: 0.1487 - val_auc: 0.6929 - val_loss: 0.1719
Epoch 9/20
3/3 - 12s - 4s/step - auc: 0.7584 - loss: 0.1412 - val_auc: 0.6929 - val_loss: 0.1718
Epoch 10/20
3/3 - 12s - 4s/step - auc: 0.8206 - loss: 0.1321 - val_auc: 0.6214 - val_loss: 0.1714
Epoch 11/20
3/3 - 15s - 5s/step - auc: 0

I0000 00:00:1790786036.806942   27590 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_218547__.48
I0000 00:00:1790786044.951131   27590 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_218547__.48


3/3 - 21s - 7s/step - auc: 0.4569 - loss: 0.1758 - val_auc: 0.5000 - val_loss: 0.1731
Epoch 2/20
3/3 - 12s - 4s/step - auc: 0.5957 - loss: 0.1663 - val_auc: 0.5000 - val_loss: 0.1729
Epoch 3/20
3/3 - 14s - 5s/step - auc: 0.6627 - loss: 0.1588 - val_auc: 0.5643 - val_loss: 0.1727
Epoch 4/20
3/3 - 12s - 4s/step - auc: 0.6435 - loss: 0.1555 - val_auc: 0.6143 - val_loss: 0.1727
Epoch 5/20
3/3 - 12s - 4s/step - auc: 0.6962 - loss: 0.1522 - val_auc: 0.6143 - val_loss: 0.1726
Epoch 6/20
3/3 - 15s - 5s/step - auc: 0.6794 - loss: 0.1535 - val_auc: 0.5643 - val_loss: 0.1726
Epoch 7/20
3/3 - 12s - 4s/step - auc: 0.6555 - loss: 0.1543 - val_auc: 0.5143 - val_loss: 0.1727
Epoch 8/20
3/3 - 15s - 5s/step - auc: 0.7105 - loss: 0.1520 - val_auc: 0.5357 - val_loss: 0.1727
Epoch 9/20
3/3 - 12s - 4s/step - auc: 0.7608 - loss: 0.1438 - val_auc: 0.5357 - val_loss: 0.1726
Epoch 10/20
3/3 - 12s - 4s/step - auc: 0.7249 - loss: 0.1454 - val_auc: 0.4357 - val_loss: 0.1727
Epoch 11/20


In [ ]:
ensemble[0].models[0].summary()

In [ ]:
i = 0
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()
target_name = target_columns[model_index]
print(f"{target_name} performance")
for p in planes:
    for j in range(2):
        df = pd.DataFrame(ensemble[i].history[target_columns.index(target_name)].history)
        df = df.rename(columns={"loss": "train_loss"})
        if "val_auc" in df.columns:
            df[["train_loss", "val_loss"]].plot(ax=axes[i])
        else:
            df[["train_loss"]].plot(ax=axes[i])

        axes[i].set_title(f"{ensemble[i].anatomical_plane}: {ensemble[i].fat_suppression} - {ensemble[i].fluid_sensitive}")
        axes[i].set_ylim(0.3, 1.0)

        axes[i].set_xlabel("Epoch")
        axes[i].set_ylabel("loss Score")
        axes[i].grid(True)
        i += 1

In [ ]:
i = 0
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()
target_name = target_columns[model_index]
print(f"{target_name} performance")
for p in planes:
    for j in range(2):
        df = pd.DataFrame(ensemble[i].history[target_columns.index(target_name)].history)
        df = df.rename(columns={"auc": "train_auc"})
        if "val_auc" in df.columns:
            df[["train_auc", "val_auc"]].plot(ax=axes[i])
        else:
            df[["train_auc"]].plot(ax=axes[i])

        axes[i].set_title(f"{ensemble[i].anatomical_plane}: {ensemble[i].fat_suppression} - {ensemble[i].fluid_sensitive}")
        axes[i].set_ylim(0.3, 1.0)
        axes[i].axhline(y=0.5, color='red', linestyle=':')

        axes[i].set_xlabel("Epoch")
        axes[i].set_ylabel("AUC Score")
        axes[i].grid(True)
        i += 1

In [ ]:
# fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(18, 10))
# axes = axes.flatten()
# plane = planes[0]
# mri_mode = 1
#
# print(f"Model: {plane} - {mri_mode}")
# for i in range(len(target_columns)):
#     model = None
#     for m in ensemble:
#         if m.anatomical_plane == plane and m.fat_suppression == mri_mode:
#             model = m
#             break
#
#     df = pd.DataFrame(model.history[i].history)
#     df = df.rename(columns={"loss": "train_loss"})
#     if "val_auc" in df.columns:
#         df[["train_loss", "val_loss"]].plot(ax=axes[i])
#     else:
#         df[["train_loss"]].plot(ax=axes[i])
#
#     axes[i].set_title(f"{target_columns[i]} loss Score")
#     axes[i].set_ylim(0.3, 1.0)
#     axes[i].axhline(y=0.5, color='red', linestyle='-')
#
#     axes[i].set_ylabel("loss Score")
#     axes[i].grid(True)

In [ ]:
scores = MultiLabelModel.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

In [ ]:
data = {"name": []}
for t in target_columns:
    data[t] = []

for m in ensemble:
    data["name"].append(f"{m.anatomical_plane}_{m.fat_suppression}_{m.fluid_sensitive}")
    for i in range(len(target_columns)):
        data[target_columns[i]].append(m.auc_scores[i])

data = pd.DataFrame(data)
data[["name", target_columns[model_index]]]

In [ ]:
auc_scores = []
for t in target_columns:
    pos = data[t].argmax()
    auc_scores.append(data[t].iloc[pos])
    print(f"{t}: score -> {data[t].iloc[pos]:.3f}, model -> {data['name'].iloc[pos]}")
print(f"Average: {np.mean(auc_scores)}")

In [ ]:
unlabeled = pd.read_csv("train.csv").drop(columns=["Report"])
unlabeled = unlabeled[unlabeled["StudyInstanceUID"].isin(study_ids)]
unlabeled = unlabeled[unlabeled.isnull().any(axis=1)]
unlabeled.head()

In [ ]:
# labels = MultiLabelModel.make_prediction(unlabeled["StudyInstanceUID"], ensemble, 2)
# labels